In [0]:
# ============================================================
# STEP 1: Configuration
# ============================================================
REDPANDA_BROKER = "d8dfctlqfk2v0e5r4290.any.us-east-1.mpx.prd.cloud.redpanda.com:9092"
TOPIC = "raw_events"
USERNAME = "nalluriravi"
PASSWORD = "Tesco@123"

JAAS_CONFIG = (
    f'kafkashaded.org.apache.kafka.common.security.scram.ScramLoginModule required '
    f'username="{USERNAME}" password="{PASSWORD}";'
)

# ============================================================
# STEP 2: Read Stream from Redpanda
# ============================================================
raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", REDPANDA_BROKER)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")        # Use "latest" in production
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "SCRAM-SHA-256")
    .option("kafka.sasl.jaas.config", JAAS_CONFIG)
    .option("kafka.ssl.endpoint.identification.algorithm", "https")
    .option("failOnDataLoss", "false")
    .load()
)

print("✅ Stream connected to Redpanda")
print(raw_stream)

In [0]:
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# ============================================================
# STEP 3: Define Schema (match your message structure)
# ============================================================
schema = StructType([
    StructField("customer_id",   IntegerType(), True),
    StructField("customer_name", StringType(),  True),
    StructField("city",          StringType(),  True)
])

# ============================================================
# STEP 4: Parse JSON + Add metadata columns
# ============================================================
parsed_stream = (
    raw_stream
    .select(
        col("offset"),
        col("partition"),
        col("timestamp").alias("event_time"),
        from_json(col("value").cast("string"), schema).alias("data")
    )
    .select(
        "offset",
        "partition",
        "event_time",
        "data.customer_id",
        "data.customer_name",
        "data.city",
        current_timestamp().alias("processed_at")
    )
)

print("✅ Schema parsed successfully")


In [0]:
# write to delta table
query = (
    parsed_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/workspace/default/kafka/redpanda_source")
    .trigger(availableNow=True)
    .toTable("raw_events_delta")   # Creates table in Databricks
)

query.awaitTermination()



In [0]:
spark.read.table("raw_events_delta").display()